# 埃尔米特多项式离线计算器（非线性回归的线性拟合）

**第一步：选择约定（两套权函数与正交基）**

| 约定 | 权函数 | 正交基 | 递推关系 |
|---|---|---|---|
| 物理学家版 | $f(x)=e^{-x^2}$ | 物理学家版 $\{H_n\}$ | $H_0=1,\ H_1=2x,\ H_{n+1}=2xH_n-2nH_{n-1}$ |
| 概率学家版 | $\varphi(x)=\dfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$（标准正态密度） | 概率学家版 $\{He_n\}$ | $He_0=1,\ He_1=x,\ He_{n+1}=xHe_n-nHe_{n-1}$ |

两套基的换算关系：$He_n(x)=2^{-n/2}H_n\!\left(\dfrac{x}{\sqrt2}\right)$，还原的是同一个函数。

**标准称呼对照**

- 内积：物理学家版 $\langle p,q\rangle_f=\displaystyle\int_{-\infty}^{\infty}p(x)q(x)e^{-x^2}\,dx$；
  概率学家版 $\langle p,q\rangle_\varphi=\displaystyle\int_{-\infty}^{\infty}p(x)q(x)\varphi(x)\,dx$
- 你说的“函数向量的系数基”，标准名称是 **埃尔米特展开系数（广义傅里叶系数）**：
  $\alpha_n=\dfrac{\langle h,P_n\rangle_w}{\langle P_n,P_n\rangle_w}$，于是 $h(x)=\displaystyle\sum_{n=0}^{N}\alpha_n P_n(x)$
  （$P_n$ 按约定取 $H_n$ 或 $He_n$）
- Rodrigues 记号：物理学家版 $\dfrac{f^{(n)}}{f}=(-1)^n H_n$；
  概率学家版 $\dfrac{\varphi^{(n)}}{\varphi}=(-1)^n He_n$，该基下的系数向量本工具同时给出

**两种模式（对应两个 Tab）**

| | 输入 | 对应 Cell | 方法 |
|---|---|---|---|
| 模式一 | 已知 $h$：输入阶数 $d$ 与各阶系数 $a_k$ | Cell 4 | 精确有理运算，输出分数形式的 $\alpha_n$ 与系数向量 |
| 模式二 | 未知 $h$：输入 $m$ 个点坐标 $(x_i,y_i)$ 与最高阶 $N$ | Cell 5 | 笔记公式 III 梯形数值积分，输出完整多项式方程 |

所有公式均以 LaTeX 规范排版，分数为上下堆叠形式；模式二的分数为小数的近似分数（分母上限 10000）。

In [1]:
# ===== Cell 2｜依赖导入与全局配置（无需修改，直接运行）=====
import re
import numpy as np
import sympy as sp
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Latex, Markdown, clear_output

x = sp.symbols('x')
%matplotlib inline

In [2]:
# ===== Cell 3｜核心数学函数（双约定参数化，无需修改）=====

# ---------- 约定配置：a 为递推尺度（物理学家版 2，概率学家版 1）----------
CONV = {
    'phys': {
        'a': 2,
        'weight_num': lambda xv: np.exp(-xv ** 2),
        'bname': 'H', 'wname': 'f',
        'inner_tex': r'\int_{-\infty}^{\infty}p(x)q(x)e^{-x^2}\,dx',
        'weight_tex': r'e^{-x^2}',
    },
    'prob': {
        'a': 1,
        'weight_num': lambda xv: np.exp(-xv ** 2 / 2.0),
        'bname': 'He', 'wname': r'\varphi',
        'inner_tex': r'\int_{-\infty}^{\infty}p(x)q(x)\frac{1}{\sqrt{2\pi}}e^{-x^2/2}\,dx',
        'weight_tex': r'\frac{1}{\sqrt{2\pi}}e^{-x^2/2}',
    },
}
CONV_NAME = 'phys'

# ---------- 精确模式 ----------
def parse_rational(s):
    """解析整数、小数、分数（如 7/2、-1/4），返回精确有理数 sp.Rational"""
    s = str(s).strip().replace(' ', '')
    if '/' in s:
        p, q = s.split('/')
        return sp.Rational(int(p), int(q))
    return sp.Rational(s)

def exact_basis_poly(n, conv_name):
    """递推构造第 n 阶基多项式（精确）：P_{k+1} = a x P_k - a k P_{k-1}"""
    a = sp.Integer(CONV[conv_name]['a'])
    if n == 0:
        return sp.Integer(1)
    p_prev, p_cur = sp.Integer(1), a * x
    for k in range(1, n):
        p_prev, p_cur = p_cur, sp.expand(a * x * p_cur - a * k * p_prev)
    return p_cur

def basis_matrix(d, conv_name):
    """M[k,n] = P_n 中 x^k 的系数；普通系数 a 与展开系数 alpha 满足 a = M alpha"""
    M = sp.zeros(d + 1, d + 1)
    for n in range(d + 1):
        for monom, c in sp.Poly(exact_basis_poly(n, conv_name), x).terms():
            M[monom[0], n] = c
    return M

def exact_expansion(coeffs, conv_name):
    """输入普通多项式升幂系数 [a0,...,ad]，返回 alpha（P_n 基）、beta（Rodrigues 基）、h 表达式"""
    d = len(coeffs) - 1
    alpha = [sp.simplify(v) for v in basis_matrix(d, conv_name).solve(sp.Matrix(coeffs))]
    beta = [((-1) ** n) * v for n, v in enumerate(alpha)]
    h_expr = sp.expand(sum(coeffs[k] * x ** k for k in range(d + 1)))
    return alpha, beta, h_expr

# ---------- 数值模式 ----------
def basis_coeff_arrays(N, conv_name):
    """递推构造 P_0...P_N 的升幂系数数组（float）"""
    a = float(CONV[conv_name]['a'])
    P = [np.array([1.0]), np.array([0.0, a])]
    for n in range(1, N):
        shifted = np.concatenate(([0.0], a * P[n]))            # a x P_n
        prev_pad = np.zeros(len(shifted))
        prev_pad[:len(P[n - 1])] = P[n - 1]
        P.append(shifted - a * n * prev_pad)
    return P[:N + 1]

def basis_values(xv, N, conv_name):
    """P_n 在节点 xv 上的取值，返回 (N+1) x m 数组"""
    arrays = basis_coeff_arrays(N, conv_name)
    return np.vstack([np.polyval(p[::-1], xv) for p in arrays])

def trapezoid_alpha(xv, yv, N, conv_name):
    """笔记公式 III：在节点区间 [x_1,x_m] 上用梯形法数值计算 alpha_0...alpha_N"""
    Pv = basis_values(xv, N, conv_name)
    weight = CONV[conv_name]['weight_num'](xv)
    dx = np.diff(xv)
    alpha = []
    for n in range(N + 1):
        g_num = yv * Pv[n] * weight
        g_den = Pv[n] ** 2 * weight
        num = np.sum(dx / 2.0 * (g_num[1:] + g_num[:-1]))
        den = np.sum(dx / 2.0 * (g_den[1:] + g_den[:-1]))
        alpha.append(num / den)
    return np.array(alpha, dtype=float), Pv

def expand_numeric(alpha, conv_name):
    """把 sum alpha_n P_n 合并为普通多项式升幂系数"""
    arrays = basis_coeff_arrays(len(alpha) - 1, conv_name)
    c = np.zeros(max(len(p) for p in arrays))
    for n, a in enumerate(alpha):
        c[:len(arrays[n])] += a * arrays[n]
    return c

def approx_frac(v, max_den=10000):
    """小数转近似分数，用于规范 LaTeX 显示"""
    return sp.Rational(float(v)).limit_denominator(max_den)

def _in_fixed_range(v):
    """常规量级判定：此范围内才显示定点小数与近似分数"""
    av = abs(float(v))
    return v != 0 and av < 1e5 and av >= 1e-4

def _sci_tex(v):
    """科学计数法的 LaTeX 形式，如 2.711\\times 10^{4}"""
    mant, exp = ('%.3e' % float(v)).split('e')
    return r'%s\times 10^{%d}' % (mant, int(exp))

def fmt_num(v, fixed=6, latex=False):
    """常规量级用定点小数；过大（≥1e5）或过小（<1e-4）用科学计数法，3 位有效尾数"""
    if float(v) == 0:
        return '0'
    if not _in_fixed_range(v):
        return _sci_tex(v) if latex else '%.3e' % float(v)
    return ('%.' + str(fixed) + 'f') % float(v)

def _coef_parts(v, approx=True):
    """系数拆为 (负号, 主体 LaTeX)；常规量级用近似分数，超范围用科学计数法；零返回 None"""
    if float(v) == 0:
        return None
    if _in_fixed_range(v):
        c = approx_frac(v) if approx else sp.simplify(v)
        return _split_sign(c)
    return float(v) < 0, fmt_num(abs(float(v)), latex=True)

def L(s):
    """把裸 LaTeX 包进 gathered 环境，保证前端按公式渲染而非显示源码"""
    return Latex(r'\begin{gathered} ' + s + r' \end{gathered}')

def _split_sign(c):
    """拆出系数的正负号与主体 LaTeX，避免替换字符串产生双空格"""
    s = sp.latex(c)
    if s.startswith('-'):
        return True, s[1:].lstrip()
    return False, s

def _join_signed(signed_terms):
    """signed_terms 为 (是否负号, 主体 LaTeX) 列表，按规范正负号拼接"""
    if not signed_terms:
        return '0'
    neg, body = signed_terms[0]
    s = '-' + body if neg else body
    for neg, body in signed_terms[1:]:
        s += (' - ' if neg else ' + ') + body
    return s

def latex_basis_form(alpha, conv_name, approx=True):
    """alpha 序列转 sum alpha_n P_n 的 LaTeX"""
    bn = CONV[conv_name]['bname']
    signed = []
    for n, v in enumerate(alpha):
        parts = _coef_parts(v, approx)
        if parts is None:
            continue
        neg, body = parts
        signed.append((neg, body + r'\,%s_{%d}(x)' % (bn, n)))
    return _join_signed(signed)

def latex_standard_poly(c_asc, approx=True):
    """升幂系数转标准多项式 LaTeX"""
    signed = []
    for k, v in enumerate(c_asc):
        parts = _coef_parts(v, approx)
        if parts is None:
            continue
        neg, body = parts
        if k == 0:
            tail = ''
        elif k == 1:
            tail = r'\,x'
        else:
            tail = r'\,x^{%d}' % k
        signed.append((neg, body + tail))
    return _join_signed(signed)

def rodrigues_tex(conv_name):
    """Rodrigues 恒等式的 LaTeX"""
    if conv_name == 'phys':
        return r'\frac{f^{(n)}}{f}=(-1)^n H_n'
    return r'\frac{\varphi^{(n)}}{\varphi}=(-1)^n He_n'

def static_trap_tex(conv_name):
    """笔记公式 III 完整 LaTeX（按约定替换基符号与权函数）"""
    bn = CONV[conv_name]['bname']
    if conv_name == 'phys':
        w = lambda i: r'e^{-x_{%s}^2}' % i
    else:
        w = lambda i: r'\frac{1}{\sqrt{2\pi}}e^{-x_{%s}^2/2}' % i
    return (
        r'\alpha_n \approx \frac{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[y_{i+1}%s_n(x_{i+1})%s+y_i%s_n(x_i)%s\right]}{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[%s_n^2(x_{i+1})%s+%s_n^2(x_i)%s\right]}'
        % (bn, w('i+1'), bn, w('i'), bn, w('i+1'), bn, w('i')))

def order_warning(m, N):
    """三级阶数提示：返回 (level, text)，level 取 ok / caution / danger（p=N+1 为待估参数数）"""
    p = N + 1
    if m < p:
        return 'danger', ('待估参数 %d 个多于点数 %d，问题欠定、解不唯一，'
                          '以下结果仅作演示，不具统计意义' % (p, m))
    if m < 2 * p:
        return 'caution', ('当前 %d 个点估计 %d 个系数，每参数不足 2 个点，'
                           '系数可能不稳定，请谨慎引用' % (m, p))
    return 'ok', ''

_WARN_STYLE = {
    'caution': ('#fef3c7', '#f59e0b', '#92400e'),
    'danger': ('#fee2e2', '#ef4444', '#991b1b'),
}

def warning_banner(m, N):
    """生成黄/红提示横幅 Markdown；ok 级别返回空字符串"""
    level, text = order_warning(m, N)
    if level == 'ok':
        return ''
    bg, border, color = _WARN_STYLE[level]
    return Markdown(
        '<div style="background:%s;border:1px solid %s;color:%s;border-radius:10px;'
        'padding:10px 14px;margin:10px 0;font-size:13.5px">%s</div>'
        % (bg, border, color, text))

In [3]:
# ===== Cell 4｜模式一：已知 h(x)，输入阶数 d 与各阶系数 a_k，运行本 Cell 出界面 =====
conv_select1 = widgets.ToggleButtons(
    options=[('物理学家版 H_n（权 f=e^{-x²}）', 'phys'),
             ('概率学家版 He_n（权 φ=标准正态）', 'prob')],
    value=CONV_NAME, description='约定选择', style={'description_width': 'initial'})

def _on_conv1(change):
    global CONV_NAME
    CONV_NAME = change['new']
    if 'conv_select2' in globals() and conv_select2.value != change['new']:
        conv_select2.value = change['new']

conv_select1.observe(_on_conv1, names='value')

degree_box = widgets.BoundedIntText(value=2, min=0, max=12, description='阶数 d')
coeff_box = widgets.VBox()

def rebuild_coeff_inputs(_=None):
    coeff_box.children = [
        widgets.HBox([
            widgets.Label('a_%d（x^%d 项系数）：' % (k, k)),
            widgets.Text(value='0', layout=widgets.Layout(width='150px'))
        ]) for k in range(degree_box.value + 1)
    ]

degree_box.observe(rebuild_coeff_inputs, names='value')
rebuild_coeff_inputs()
mode1_out = widgets.Output()

def fill_mode1(vals):
    """一键填充例题：vals 为升幂系数字符串列表"""
    degree_box.value = len(vals) - 1
    rebuild_coeff_inputs()
    for k, v in enumerate(vals):
        coeff_box.children[k].children[1].value = v

def run_mode1(_):
    with mode1_out:
        clear_output(wait=True)
        try:
            coeffs = [parse_rational(row.children[1].value) for row in coeff_box.children]
            if all(c == 0 for c in coeffs):
                raise ValueError('系数不能全为 0')
            conv = conv_select1.value
            cfg = CONV[conv]
            bn, wn = cfg['bname'], cfg['wname']
            alpha, beta, h_expr = exact_expansion(coeffs, conv)
            d = len(coeffs) - 1
            display(L(r'输入函数：\quad h(x) = ' + sp.latex(h_expr)))
            basis = r' \\ '.join(
                r'%s_{%d}(x) = %s' % (bn, n, sp.latex(exact_basis_poly(n, conv)))
                for n in range(d + 1))
            display(Latex(r'使用的正交基：$\begin{align*} %s \end{align*}$' % basis))
            display(L(
                r'\alpha_n = \frac{\langle h,%s_n\rangle_{%s}}{\langle %s_n,%s_n\rangle_{%s}},'
                r'\qquad \langle p,q\rangle_{%s} = %s'
                % (bn, wn, bn, bn, wn, wn, cfg['inner_tex'])))
            lines = r' \\ '.join(
                r'\alpha_{%d} = %s' % (n, sp.latex(alpha[n])) for n in range(d + 1))
            display(Latex(
                r'埃尔米特展开系数（广义傅里叶系数）：$\begin{align*} %s \end{align*}$' % lines))
            display(L(r'\therefore\quad h(x) = ' + latex_basis_form(alpha, conv, approx=False)))
            avec = r' \\ '.join(sp.latex(v) for v in alpha)
            bvec = r' \\ '.join(sp.latex(v) for v in beta)
            display(L(
                r'系数向量：\quad F_{%s}=\begin{bmatrix} %s \end{bmatrix}\quad \text{（}%s_n\text{ 基）},'
                r'\qquad %s,\ F=\begin{bmatrix} %s \end{bmatrix}'
                % (bn, avec, bn, rodrigues_tex(conv), bvec)))
        except Exception as e:
            display(Markdown('<span style="color:#c0392b">输入有误：%s</span>' % e))

btn1 = widgets.Button(description='计算埃尔米特展开', button_style='primary')
btn1.on_click(run_mode1)
demo1a = widgets.Button(description='例题：x²+2x+3')
demo1a.on_click(lambda _: (fill_mode1(['3', '2', '1']), run_mode1(None)))
demo1b = widgets.Button(description='例题：9x⁴-7x³+6x²-11x+3')
demo1b.on_click(lambda _: (fill_mode1(['3', '-11', '6', '-7', '9']), run_mode1(None)))

mode1_ui = widgets.VBox([
    conv_select1,
    widgets.Label('h(x) = a_0 + a_1 x + … + a_d x^d，系数可填整数、小数或分数（如 7/2）'),
    degree_box,
    coeff_box,
    widgets.HBox([btn1, demo1a, demo1b]),
    mode1_out
])

In [4]:
# ===== Cell 5｜模式二：h 未知，输入 m 个点坐标与最高阶 N，运行本 Cell 出界面 =====
conv_select2 = widgets.ToggleButtons(
    options=[('物理学家版 H_n（权 f=e^{-x²}）', 'phys'),
             ('概率学家版 He_n（权 φ=标准正态）', 'prob')],
    value=CONV_NAME, description='约定选择', style={'description_width': 'initial'})

def _on_conv2(change):
    global CONV_NAME
    CONV_NAME = change['new']
    if 'conv_select1' in globals() and conv_select1.value != change['new']:
        conv_select1.value = change['new']

conv_select2.observe(_on_conv2, names='value')

demo_points = ('0.13, 2.417\n0.47, 3.186\n1.26, 4.732\n'
               '1.83, 4.105\n3.14, 2.891\n4.72, 1.634')
points_area = widgets.Textarea(value=demo_points,
                               layout=widgets.Layout(width='340px', height='190px'))
order_box = widgets.BoundedIntText(value=4, min=0, max=20, description='最高阶 N')
mode2_out = widgets.Output()

def parse_points(text):
    """解析每行一个点：支持英文逗号、中文逗号或空白分隔，自动按 x 排序"""
    pts = []
    for line in text.strip().splitlines():
        line = line.replace('，', ',').strip()
        if not line:
            continue
        parts = re.split(r'[,\s]+', line)
        if len(parts) != 2:
            raise ValueError('无法解析的行：“%s”，每行需为 x, y 两个数' % line)
        pts.append((float(parts[0]), float(parts[1])))
    if len(pts) < 2:
        raise ValueError('至少需要 2 个点')
    pts.sort(key=lambda t: t[0])
    xv = np.array([p[0] for p in pts])
    if len(set(xv.tolist())) != len(pts):
        raise ValueError('存在重复的 x 值')
    return xv, np.array([p[1] for p in pts])

def run_mode2(_):
    with mode2_out:
        clear_output(wait=True)
        try:
            xv, yv = parse_points(points_area.value)
            N = order_box.value
            m = len(xv)
            conv = conv_select2.value
            banner = warning_banner(m, N)
            if banner:
                display(banner)
            alpha, Pv = trapezoid_alpha(xv, yv, N, conv)
            c = expand_numeric(alpha, conv)
            yfit = sum(alpha[n] * Pv[n] for n in range(N + 1))
            display(L(static_trap_tex(conv)))
            display(Markdown('数值积分在区间 [%.3g, %.3g] 上近似全空间积分；共 %d 个点、%d 段。'
                             % (xv[0], xv[-1], m, m - 1)))
            def alpha_row(n):
                v = alpha[n]
                if _in_fixed_range(v):
                    head = r'\alpha_{%d} &= %s \approx %s' % (
                        n, fmt_num(v), sp.latex(approx_frac(v)))
                else:
                    head = r'\alpha_{%d} &= %s' % (n, fmt_num(v, latex=True))
                if n >= 1:
                    d = float(v) - float(alpha[n - 1])
                    head += r', & \Delta\alpha_{%d} &= %s' % (
                        n, fmt_num(d, latex=True))
                return head
            rows = r' \\ '.join(alpha_row(n) for n in range(N + 1))
            display(Latex(r'$\begin{aligned} %s \end{aligned}$' % rows))
            sse = float(np.sum((yv - yfit) ** 2))
            display(L(r'\mathrm{SSE} = \sum_{i=1}^{m}(y_i-F(x_i))^2 = '
                      + fmt_num(sse, latex=True)))
            display(L(r'F(x) = ' + latex_basis_form(alpha, conv)
                      + r'\quad \text{（埃尔米特形式，分数为近似值）}'))
            display(L(r'F(x) = ' + latex_standard_poly(c)
                      + r'\quad \text{（标准多项式形式，分数为近似值）}'))
            tbl = '| i | x_i | y_i | F(x_i) | 残差 |\n|---|---|---|---|---|\n'
            for i in range(m):
                tbl += '| %d | %.4g | %.4g | %.4g | %.4g |\n' % (
                    i + 1, xv[i], yv[i], yfit[i], yv[i] - yfit[i])
            display(Markdown(tbl))
            grid = np.linspace(xv[0], xv[-1], 400)
            Pg = basis_values(grid, N, conv)
            yg = sum(alpha[n] * Pg[n] for n in range(N + 1))
            plt.figure(figsize=(7, 3.4))
            plt.plot(grid, yg, label='F(x)')
            plt.scatter(xv, yv, color='crimson', zorder=3, label='data points')
            plt.xlabel('x'); plt.ylabel('y'); plt.legend()
            plt.grid(alpha=.3); plt.tight_layout()
            plt.show()
        except Exception as e:
            display(Markdown('<span style="color:#c0392b">输入有误：%s</span>' % e))

btn2 = widgets.Button(description='计算完整多项式', button_style='primary')
btn2.on_click(run_mode2)

mode2_ui = widgets.VBox([
    conv_select2,
    widgets.Label('每行输入一个点，格式 x, y（支持空格或中文逗号分隔）'),
    points_area,
    order_box,
    btn2,
    mode2_out
])

tab = widgets.Tab(children=[mode1_ui, mode2_ui])
tab.titles = ('模式一：已知 h 多项式', '模式二：已知数据点')
display(tab)

In [5]:
# ===== Cell 6｜离线自检：两约定对照例题，可随时重跑 =====
# --- 物理学家版：h=x^2+2x+3 -> F(7/2,-1,1/4) ---
alpha1, beta1, h1 = exact_expansion(
    [sp.Rational(3), sp.Rational(2), sp.Rational(1)], 'phys')
assert beta1 == [sp.Rational(7, 2), sp.Rational(-1), sp.Rational(1, 4)]
assert alpha1 == [sp.Rational(7, 2), sp.Rational(1), sp.Rational(1, 4)]

# --- 概率学家版：同一函数 -> F_He(4,2,1)，Rodrigues 基 (4,-2,1) ---
alpha1p, beta1p, _ = exact_expansion(
    [sp.Rational(3), sp.Rational(2), sp.Rational(1)], 'prob')
assert alpha1p == [sp.Rational(4), sp.Rational(2), sp.Rational(1)]
assert beta1p == [sp.Rational(4), sp.Rational(-2), sp.Rational(1)]

# --- 9x^4 例题：两约定均可无损还原 ---
coeffs4 = [sp.Rational(3), sp.Rational(-11), sp.Rational(6),
           sp.Rational(-7), sp.Rational(9)]
for conv in ('phys', 'prob'):
    a2, _, h2 = exact_expansion(coeffs4, conv)
    rebuilt = sp.expand(sum(a2[n] * exact_basis_poly(n, conv) for n in range(5)))
    assert rebuilt == h2

# --- 稠密点梯形法：两约定均逼近精确系数（概率学家版尾部更重，网格取 ±5）---
xg = np.linspace(-5, 5, 501)
yg = xg ** 2 + 2 * xg + 3
al_phys, _ = trapezoid_alpha(xg, yg, 2, 'phys')
err_phys = float(np.abs(al_phys - np.array([3.5, 1.0, 0.25])).max())
assert err_phys < 1e-3
al_prob, _ = trapezoid_alpha(xg, yg, 2, 'prob')
err_prob = float(np.abs(al_prob - np.array([4.0, 2.0, 1.0])).max())
assert err_prob < 1e-3

# --- 三级阶数提示逻辑 ---
assert order_warning(6, 4)[0] == 'caution'     # 6 点估计 5 参数
assert order_warning(6, 9)[0] == 'danger'      # 参数多于点数，欠定
assert order_warning(101, 20)[0] == 'ok'       # 稠密点，无提示
assert warning_banner(101, 20) == ''

# --- Δα 一阶差分与 SSE 校验（稠密网格，物理学家版）---
d_phys = np.diff(al_phys)
assert np.allclose([d_phys[n - 1] for n in range(1, len(al_phys))],
                   [al_phys[n] - al_phys[n - 1] for n in range(1, len(al_phys))])
assert np.isclose(d_phys.sum(), al_phys[-1] - al_phys[0])   # 望远镜求和
P_g = basis_values(xg, 2, 'phys')
yfit_g = sum(al_phys[n] * P_g[n] for n in range(3))
sse_g = float(np.sum((yg - yfit_g) ** 2))
assert sse_g >= 0

print('自检通过：')
print('  1. 物理学家版 h=x^2+2x+3 -> F(7/2, -1, 1/4)，与笔记图 2 完全一致')
print('  2. 概率学家版 h=x^2+2x+3 -> F_He(4, 2, 1)，Rodrigues 基 (4, -2, 1)')
print('  3. 9x^4 例题两约定均可无损还原原函数')
print('  4. 稠密点梯形法最大偏差：物理学家版 %.2e，概率学家版 %.2e（均 < 1e-3）'
      % (err_phys, err_prob))
print('  5. 三级阶数提示（黄/红/无）分级逻辑正确，高阶不再硬拦截')
print('  6. Δα 一阶差分望远镜求和成立，SSE 计算正常（稠密网格 SSE=%.2e）' % sse_g)
display(Latex(
    r'h(x)=x^2+2x+3=\frac{7}{2}H_0+H_1+\frac{1}{4}H_2'
    r'=4He_0+2He_1+He_2'))

自检通过：
  1. 物理学家版 h=x^2+2x+3 -> F(7/2, -1, 1/4)，与笔记图 2 完全一致
  2. 概率学家版 h=x^2+2x+3 -> F_He(4, 2, 1)，Rodrigues 基 (4, -2, 1)
  3. 9x^4 例题两约定均可无损还原原函数
  4. 稠密点梯形法最大偏差：物理学家版 6.88e-11，概率学家版 2.98e-05（均 < 1e-3）
  5. 三级阶数提示（黄/红/无）分级逻辑正确，高阶不再硬拦截
  6. Δα 一阶差分望远镜求和成立，SSE 计算正常（稠密网格 SSE=4.56e-15）


<IPython.core.display.Latex object>